# Prompt Engineering Techniques Overview

Source slide: `slides/prompt-engineering/04_Prompt_Engineering_Techniques.md`

This notebook gives one compact executable example for each technique family named in the overview slide.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Basic Techniques

**Failure mode:** Basic techniques improve raw instruction quality. If the task is underspecified, the model is forced to improvise.

**Failure test:** The failure prompt asks for churn analysis without defining the output shape or grounding data, so the answer can become generic business language.

**Technique:** Use explicit instructions, grounded context, and clear output formatting.

**Improved example:** The improved prompt names the deliverable, the number of findings, and the evidence requirement, which sharply narrows the solution space.


In [ ]:
await run_prompt_pair(
    technique_name='Basic Techniques',
    failure_prompt='Explain why customers are churning.',
    improved_prompt='Using the churn notes below, list the top two churn drivers with one cited phrase for each.\n\nNotes:\n- 41% of lost accounts mentioned onboarding delays.\n- 28% mentioned reporting exports timing out.\n- 9% mentioned pricing confusion.',
    failure_test='The failure prompt asks for churn analysis without defining the output shape or grounding data, so the answer can become generic business language.',
    fix_explanation='The improved prompt names the deliverable, the number of findings, and the evidence requirement, which sharply narrows the solution space.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Reasoning frameworks & Cognitive Architectures

**Failure mode:** Reasoning frameworks matter when the model has to work through multiple dependent steps instead of answering from a single fact.

**Failure test:** The failure prompt asks for a schedule calculation but gives no structure for intermediate reasoning, which increases the chance of arithmetic or calendar mistakes.

**Technique:** Require the model to reason through intermediate steps before producing the final answer.

**Improved example:** The improved prompt explicitly asks for step-by-step reasoning, which encourages the model to surface and check its logic instead of jumping straight to a conclusion.


In [ ]:
await run_prompt_pair(
    technique_name='Reasoning frameworks & Cognitive Architectures',
    failure_prompt='If a project starts on Tuesday and takes 7 business days, when does it end?',
    improved_prompt='If a project starts on Tuesday and takes 7 business days, determine the end date. Think step by step, count only business days, and then give the final day on a separate line.',
    failure_test='The failure prompt asks for a schedule calculation but gives no structure for intermediate reasoning, which increases the chance of arithmetic or calendar mistakes.',
    fix_explanation='The improved prompt explicitly asks for step-by-step reasoning, which encourages the model to surface and check its logic instead of jumping straight to a conclusion.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Retrieval Augmented Generation

**Failure mode:** RAG matters when the answer depends on source material the model may not know or should not guess.

**Failure test:** The failure prompt asks a policy question with no policy text attached, so the model may hallucinate a plausible company rule.

**Technique:** Attach the relevant source and instruct the model to ground its answer in that material.

**Improved example:** The improved prompt includes the policy excerpt and tells the model to answer only from that context, which lowers hallucination risk.


In [ ]:
await run_prompt_pair(
    technique_name='Retrieval Augmented Generation',
    failure_prompt='What is our refund window for annual plans?',
    improved_prompt='Answer using only the policy excerpt below. If the answer is missing, say so.\n\nPolicy excerpt:\n- Annual plans can be refunded within 30 calendar days of purchase.\n- Monthly plans can be refunded within 7 calendar days of purchase.\n\nQuestion: What is our refund window for annual plans?',
    failure_test='The failure prompt asks a policy question with no policy text attached, so the model may hallucinate a plausible company rule.',
    fix_explanation='The improved prompt includes the policy excerpt and tells the model to answer only from that context, which lowers hallucination risk.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Parameter tuning (temperature, top_p)

**Failure mode:** Sampling controls change how stable or creative an answer is. New prompt engineers often use one configuration for every task, which leads to drift on deterministic tasks or blandness on creative tasks.

**Failure test:** This notebook uses the real Copilot SDK with ambient auth, but the public session API does not expose temperature or top_p directly. The failure prompt is the kind of prompt that drifts when those settings are left too random for extraction work.

**Technique:** Match sampling settings to the task: low variance for extraction, higher variance for ideation. Use the same prompt while changing the provider setting in an environment that exposes it.

**Improved example:** The improved prompt narrows the task and the configuration note tells you which lower-variance settings to apply when your Copilot host exposes them.

**Configuration note:** Recommended tuned run for this task: use temperature=0 and top_p=1 in the Copilot host or provider settings when available, then rerun the improved prompt.


In [ ]:
await run_prompt_pair(
    technique_name='Parameter tuning (temperature, top_p)',
    failure_prompt='Read this incident note and return the single severity label: “Login failed for all customers in production for 18 minutes.”',
    improved_prompt='Return exactly one label from {SEV1, SEV2, SEV3}. Incident note: “Login failed for all customers in production for 18 minutes.”',
    failure_test='This notebook uses the real Copilot SDK with ambient auth, but the public session API does not expose temperature or top_p directly. The failure prompt is the kind of prompt that drifts when those settings are left too random for extraction work.',
    fix_explanation='The improved prompt narrows the task and the configuration note tells you which lower-variance settings to apply when your Copilot host exposes them.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
    config_note='Recommended tuned run for this task: use temperature=0 and top_p=1 in the Copilot host or provider settings when available, then rerun the improved prompt.',
)


## Constrained Decoding

**Failure mode:** Constrained decoding matters when the output must be safe for code, policy, or automation systems to consume.

**Failure test:** The failure prompt asks for structured extraction but never defines a schema, so the model can mix prose with data and break downstream parsing.

**Technique:** State the exact schema or constraint the response must satisfy.

**Improved example:** The improved prompt supplies a JSON shape and asks for only that shape, which makes the result much easier to validate and consume.


In [ ]:
await run_prompt_pair(
    technique_name='Constrained Decoding',
    failure_prompt='Extract the customer contact details from this text: “Jamie Rivera can be reached at jamie@example.com and 555-0101.”',
    improved_prompt='Extract the customer contact details from the text and return JSON only in this shape: {"name": string, "email": string, "phone": string}. Text: “Jamie Rivera can be reached at jamie@example.com and 555-0101.”',
    failure_test='The failure prompt asks for structured extraction but never defines a schema, so the model can mix prose with data and break downstream parsing.',
    fix_explanation='The improved prompt supplies a JSON shape and asks for only that shape, which makes the result much easier to validate and consume.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
